## Secrets — and why env vars aren't secret

Every tutorial does this, and it's wrong for real credentials:

```bash
docker run -e DB_PASSWORD=hunter2 myorg/api
```

Convenient — **not secret.** That password leaks in at least four places:

1. **`docker inspect`** prints the whole env block — anyone with daemon access reads it.
2. **`/proc/<pid>/environ`** on the host exposes it to anyone who can read it.
3. **Shell history** on the machine that launched it.
4. **Crash dumps and logs** that helpfully capture "the environment at crash time."

And env vars are **inherited by every child process**, so each shell-out inside the container sees the password too. Env vars are for *config* (log level, feature flags), not for *secrets*.

**The container-native way: a secret file** mounted at a known path, readable only by the app's user. In Compose:

```yaml
services:
  api:
    secrets: [db_password]
    environment:
      DB_PASSWORD_FILE: /run/secrets/db_password   # app reads the file path
secrets:
  db_password:
    file: ./secrets/db_password.txt
```

Compose mounts the file at **`/run/secrets/db_password`** with restrictive permissions, and the app reads the *path*, not an env var. Note the common convention: pass `DB_PASSWORD_FILE` (a path) rather than `DB_PASSWORD` (a value) — many official images support the `_FILE` suffix natively. The same `secrets:` syntax scales up to **Swarm secrets** (module 09), which are encrypted at rest and in transit across the cluster, and **BuildKit secret mounts** keep secrets out of image layers at *build* time (module 02). The rule: **secrets are files with tight permissions, never environment variables.**